🔷1. Create DataFrame


In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [0]:
data = [
(101,"Arjun Reddy","Hyderabad","Cardiology",5000),
(102,"Sneha Kapoor","Delhi","Orthopedics",3000),
(103,"Rahul Sharma","Mumbai","Dermatology",1500),
(104,"Priya Nair","Bangalore","Cardiology",5000),
(105,"Vikram Singh","Chennai","Neurology",7000)
]

columns = ["visit_id","patient_name","city","department","consultation_fee"]

df = spark.createDataFrame(data, columns)

display(df)

visit_id,patient_name,city,department,consultation_fee
101,Arjun Reddy,Hyderabad,Cardiology,5000
102,Sneha Kapoor,Delhi,Orthopedics,3000
103,Rahul Sharma,Mumbai,Dermatology,1500
104,Priya Nair,Bangalore,Cardiology,5000
105,Vikram Singh,Chennai,Neurology,7000


🔷2. Write Data as Parquet

In [0]:
df.write \
.mode("overwrite") \
.parquet("/tmp/patient_parquet")

🔷3. Read Parquet Data

In [0]:
parquet_df = spark.read.parquet("/tmp/patient_parquet")
display(parquet_df)

visit_id,patient_name,city,department,consultation_fee
101,Arjun Reddy,Hyderabad,Cardiology,5000
104,Priya Nair,Bangalore,Cardiology,5000
103,Rahul Sharma,Mumbai,Dermatology,1500
105,Vikram Singh,Chennai,Neurology,7000
102,Sneha Kapoor,Delhi,Orthopedics,3000


🔷4. Schema Inspection

In [0]:
parquet_df.printSchema()

root
 |-- visit_id: long (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- department: string (nullable = true)
 |-- consultation_fee: long (nullable = true)



🔷5. Column Projection (Read Specific Columns)

In [0]:
spark.read.parquet("/tmp/patient_parquet") \
.select("patient_name","city") \
.show()

🔷6. Filtering Data

In [0]:
spark.read.parquet("/tmp/patient_parquet") \
.filter("consultation_fee > 3000") \
.show()

+--------+------------+---------+----------+----------------+
|visit_id|patient_name|     city|department|consultation_fee|
+--------+------------+---------+----------+----------------+
|     101| Arjun Reddy|Hyderabad|Cardiology|            5000|
|     104|  Priya Nair|Bangalore|Cardiology|            5000|
|     105|Vikram Singh|  Chennai| Neurology|            7000|
+--------+------------+---------+----------+----------------+



🔷7. Partitioned Parquet Write

In [0]:
df.write \
.mode("overwrite") \
.partitionBy("city") \
.parquet("/tmp/patient_parquet_partitioned")

🔷8. Read Partitioned Data

In [0]:
spark.read.parquet("/tmp/patient_parquet_partitioned").show()

+--------+------------+-----------+----------------+---------+
|visit_id|patient_name| department|consultation_fee|     city|
+--------+------------+-----------+----------------+---------+
|     103|Rahul Sharma|Dermatology|            1500|   Mumbai|
|     102|Sneha Kapoor|Orthopedics|            3000|    Delhi|
|     101| Arjun Reddy| Cardiology|            5000|Hyderabad|
|     105|Vikram Singh|  Neurology|            7000|  Chennai|
|     104|  Priya Nair| Cardiology|            5000|Bangalore|
+--------+------------+-----------+----------------+---------+



🔷9. Partition Pruning (Performance Concept)

In [0]:
spark.read.parquet("/tmp/patient_parquet_partitioned") \
.filter("city = 'Hyderabad'") \
.show()

+--------+------------+----------+----------------+---------+
|visit_id|patient_name|department|consultation_fee|     city|
+--------+------------+----------+----------------+---------+
|     101| Arjun Reddy|Cardiology|            5000|Hyderabad|
+--------+------------+----------+----------------+---------+



🔷10. Append Mode

In [0]:
new_data = [
(106,"Ananya Das","Kolkata","Orthopedics",3000)
]
new_df = spark.createDataFrame(new_data, columns)
new_df.write \
.mode("append") \
.parquet("/tmp/patient_parquet")

🔷11. Overwrite Mode

In [0]:
df.write \
.mode("overwrite") \
.parquet("/tmp/patient_parquet")

🔷12. Create SQL Table on Parquet(using view, because of Unity Catalog restriction)

In [0]:
%sql
CREATE OR REPLACE VIEW patient_parquet_view AS 
SELECT * FROM parquet.`dbfs:/tmp/patient_parquet`;

🔷 13. Query Parquet (Using VIEW instead of TABLE)

In [0]:
%sql
SELECT * FROM patient_parquet_view;

visit_id,patient_name,city,department,consultation_fee
101,Arjun Reddy,Hyderabad,Cardiology,5000
104,Priya Nair,Bangalore,Cardiology,5000
103,Rahul Sharma,Mumbai,Dermatology,1500
105,Vikram Singh,Chennai,Neurology,7000
102,Sneha Kapoor,Delhi,Orthopedics,3000


🔷 14. Convert Parquet → Delta (Path-based, not table-based)

In [0]:
%sql
CONVERT TO DELTA parquet.`dbfs:/tmp/patient_parquet`;

🔷 15. Compare Parquet vs Delta

In [0]:
%sql
UPDATE patient_parquet_view
SET consultation_fee = 6000
WHERE visit_id = 101;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7040856870446902>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'UPDATE patient_parquet_view\nSET consultation_fee = 6000\nWHERE visit_id = 101;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:213, in SqlMagic.sql(self, line, cell)
    206 except BaseException as e

In [0]:
%sql
UPDATE delta.`dbfs:/tmp/patient_parquet`
SET consultation_fee = 6000
WHERE visit_id = 101;

num_affected_rows
1
